# RoPE e posição fixa na atenção

Este notebook compara uma cabeça de self-attention sem posição, com posição fixa senoidal e com Rotary Position Embedding (RoPE).

Os embeddings e as projeções são sintéticos, determinísticos e sem treinamento. A ideia é observar a matemática da posição. Os resultados não medem compreensão de linguagem.

Execute as células na ordem. O núcleo numérico e os gráficos são os mesmos do app Streamlit.

In [ ]:
import numpy as np
from IPython.display import display
from rope import METHODS, attention, embeddings, frequencies, rotate, tokenize
from charts import attention_chart, rotation_chart

frase = "O gato viu o gato na janela."
dim = 16
posicao_inicial = 0
mascara_causal = False

tokens = tokenize(frase)
posicoes = np.arange(len(tokens)) + posicao_inicial
rotulos = [f"{p}: {t}" for p, t in zip(posicoes, tokens)]
resultado = attention(tokens, dim, posicao_inicial, mascara_causal)
list(zip(posicoes.tolist(), tokens))

## Onde a posição entra no Transformer

Uma cabeça projeta a entrada em consultas, chaves e valores:

$$Q=XW_Q,\quad K=XW_K,\quad V=XW_V$$
$$A=\operatorname{softmax}(QK^T/\sqrt{d}),\quad O=AV$$

| Método | Aplicação da posição |
| :--- | :--- |
| Sem posição | Projeta apenas o conteúdo $X$. |
| Fixa senoidal | Soma $PE(p)$ a $X$ antes das projeções de Q, K e V. |
| RoPE | Rotaciona pares de Q e K depois das projeções. V mantém apenas o conteúdo. |

Aqui mostramos o bloco de atenção. Um Transformer completo também tem múltiplas cabeças, conexões residuais, normalização e uma rede feed-forward.

Uma tabela de embeddings posicionais aprendidos é outra alternativa, ajustada durante o treino. Não há treino neste exemplo.

## A rotação

Para cada par de dimensões, $\theta_i=10000^{-2i/d}$. Na posição $p$:

$$R(p\theta_i)\begin{bmatrix}a\\b\end{bmatrix} =
\begin{bmatrix}a\cos(p\theta_i)-b\sin(p\theta_i)\\a\sin(p\theta_i)+b\cos(p\theta_i)\end{bmatrix}$$

A posição fixa usa $PE(p)_{2i}=\sin(p\theta_i)$ e $PE(p)_{2i+1}=\cos(p\theta_i)$.

O gráfico coloca **o mesmo token** em todas as posições da frase. Isso isola a posição do conteúdo. O eixo vertical indica a posição; os outros dois eixos são coordenadas reais de Q. A curva é interpolada e os pontos representam posições inteiras.

In [ ]:
token_observado = tokens[1] if len(tokens) > 1 else tokens[0]
par = 0
print(f"Token: {token_observado}; par: {2 * par}, {2 * par + 1}")
print(f"Rotação por posição: {np.degrees(frequencies(dim)[par]):.4f} graus")
rotation_chart(token_observado, dim, par, posicoes).show(renderer="plotly_mimetype")

## Mapas de atenção

Agora cada posição recebe seu token real. Cada linha é uma consulta; cada coluna é uma chave. As linhas somam 1 e todos os mapas usam a escala de 0 a 1.

Palavras repetidas têm o mesmo embedding inicial. Sem posição e sem máscara causal, elas produzem consultas iguais. A posição consegue diferenciá-las.

In [ ]:
for metodo in METHODS:
    print(metodo)
    attention_chart(resultado[metodo]["weights"], rotulos).show(renderer="plotly_mimetype")

consulta = min(1, len(tokens) - 1)
display({metodo: np.round(resultado[metodo]["weights"][consulta], 4).tolist() for metodo in METHODS})

## Posição relativa e deslocamento conjunto

A identidade central do RoPE é:

$$(R_mq)^T(R_nk)=q^TR_{n-m}k$$

O produto depende de $n-m$. Se deslocamos todas as posições pelo mesmo valor, mantendo os conteúdos, os scores e a atenção são preservados, salvo arredondamento. A soma de posição fixa com conteúdo pode mudar esses scores.

Isso não implica que toda atenção diminui com a distância, nem garante extrapolação para contextos maiores. Não há avaliação de qualidade de um modelo nesta demonstração.

In [ ]:
deslocado = attention(tokens, dim, posicao_inicial + 10, mascara_causal)
for metodo in METHODS:
    diferenca = np.max(np.abs(resultado[metodo]["weights"] - deslocado[metodo]["weights"]))
    print(f"{metodo}: maior diferença = {diferenca:.3e}")
np.testing.assert_allclose(resultado["RoPE"]["weights"], deslocado["RoPE"]["weights"], atol=1e-12)

q = embeddings(["gato"], dim)
q_rotacionado = rotate(q, np.array([7]))
np.testing.assert_allclose(np.linalg.norm(q), np.linalg.norm(q_rotacionado))
print("A rotação também preserva a norma do vetor.")

## Máscara causal e saída

Na atenção causal, cada token acessa apenas a si mesmo e os anteriores. A máscara é aplicada aos scores antes do softmax. RoPE preserva V; a mudança na atenção altera a combinação dos valores na saída da cabeça.

In [ ]:
causal = attention(tokens, dim, posicao_inicial, causal=True)["RoPE"]
np.testing.assert_allclose(causal["weights"].sum(axis=1), 1)
np.testing.assert_array_equal(np.triu(causal["weights"], 1), 0)
np.testing.assert_allclose(causal["output"], causal["weights"] @ causal["v"])
print("Saída da cabeça:", causal["output"].shape)
causal["output"].round(4)

## Para experimentar

1. Use `gato gato gato gato` para destacar a posição com conteúdo constante.
2. Altere `par` entre 0 e `dim // 2 - 1` e observe as frequências diferentes.
3. Troque a ordem das palavras e compare os mapas.
4. Altere a posição inicial sem alterar a frase.

Referências: [Attention Is All You Need](https://arxiv.org/abs/1706.03762), [RoFormer](https://arxiv.org/abs/2104.09864).